# Práctica 1 · Paso 3 — Curva de gasto y curva de duración
### Hidrología (1CIA-502) · Ingeniería Ambiental · ESPAM MFL · Unidad I
**Docente:** Ing. Cristian Paliz Acosta, MSc.

**Qué vas a producir**
1. La **curva de gasto** $Q = a\,(h-h_0)^b$ ajustada a los aforos, con su R² y sus residuos.
2. La **serie diaria de caudal**, con los días extrapolados identificados.
3. La **curva de duración** y los caudales característicos **Q10, Q50, Q90 y Q95**.

**Mano antes que modelo:** antes de ejecutar la sección 3, escribe en la celda 1 tu estimación de h₀ mirando la gráfica de los aforos.

`Archivo → Guardar una copia en Drive` y ejecuta en orden con `Shift + Enter`. Solo se edita la **celda 1**.

## 1 · CONFIGURACIÓN ✍️ (única celda que editas)

In [ ]:
ESTUDIANTE = "Apellidos Nombres"
CUENCA     = "Carrizal"   # la misma de los pasos 1 y 2
ESTACION   = "Estación de ejemplo (datos ilustrativos)"   # con tus datos: código y nombre INAMHI, p. ej. "H0294 Carrizal en Calceta"

H0_MANO = None        # tu estimación de h0 en metros, p. ej. 0.15

# Datos: por defecto, los de ejemplo del repositorio del curso.
# Si tienes los de tu estación, pon USAR_MIS_DATOS = True y sube dos CSV cuando se te pida:
#   aforos: columnas  fecha, nivel_m, caudal_m3s
#   niveles diarios: columnas  fecha, nivel_m
USAR_MIS_DATOS = False
REPO = "crispaliz/hidrologia-espam"
URL_AFOROS  = f"https://raw.githubusercontent.com/{REPO}/main/datos/aforos_ejemplo.csv"
URL_NIVELES = f"https://raw.githubusercontent.com/{REPO}/main/datos/niveles_diarios_ejemplo.csv"

## 2 · Cargar los aforos y la serie de niveles

In [ ]:
import os, io, unicodedata, numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
SLUG = unicodedata.normalize("NFKD", ESTUDIANTE).encode("ascii", "ignore").decode().replace(" ", "_")
os.makedirs("salidas", exist_ok=True)
if USAR_MIS_DATOS:
    from google.colab import files
    print("Sube primero el CSV de AFOROS:");  up = files.upload(); af = pd.read_csv(io.BytesIO(list(up.values())[0]))
    print("Ahora el CSV de NIVELES DIARIOS:"); up = files.upload(); nv = pd.read_csv(io.BytesIO(list(up.values())[0]))
else:
    af = pd.read_csv(URL_AFOROS); nv = pd.read_csv(URL_NIVELES)
af["fecha"] = pd.to_datetime(af.fecha); nv["fecha"] = pd.to_datetime(nv.fecha)
af = af.sort_values("nivel_m").reset_index(drop=True)
print(f"{len(af)} aforos · niveles de {af.nivel_m.min():.2f} a {af.nivel_m.max():.2f} m · caudales de {af.caudal_m3s.min():.3f} a {af.caudal_m3s.max():.2f} m³/s")
print(f"{len(nv)} niveles diarios · del {nv.fecha.min():%d/%m/%Y} al {nv.fecha.max():%d/%m/%Y}")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(af.nivel_m, af.caudal_m3s, "o", color="#12263A")
ax.set_xlabel("Nivel h (m)"); ax.set_ylabel("Caudal Q (m³/s)"); ax.set_xlim(0, None); ax.set_ylim(0, None)
ax.set_title("Aforos: ¿dónde cortaría la curva el eje de niveles?", fontweight="bold"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
af

## 3 · Ajuste de la curva de gasto
Con logaritmos la curva se vuelve una recta: $\log Q = \log a + b\,\log(h-h_0)$.
El cuaderno prueba valores de h₀ cada 5 mm, desde 0 hasta justo debajo del aforo más bajo, y elige el que alinea mejor los puntos (mayor R²).

In [ ]:
h, Q = af.nivel_m.values, af.caudal_m3s.values
candidatos = np.arange(0.0, h.min() - 0.004, 0.005)
res = []
for h0 in candidatos:
    x, y = np.log(h - h0), np.log(Q)
    b, lna = np.polyfit(x, y, 1)
    r2 = 1 - ((y - (lna + b * x))**2).sum() / ((y - y.mean())**2).sum()
    res.append((h0, np.exp(lna), b, r2))
res = pd.DataFrame(res, columns=["h0", "a", "b", "R2"])
mejor = res.loc[res.R2.idxmax()]
H0, A, B, R2 = mejor.h0, mejor.a, mejor.b, mejor.R2
def gasto(hh):
    hh = np.asarray(hh, dtype=float)
    return np.where(hh > H0, A * np.clip(hh - H0, 0, None)**B, 0.0)

print(f"Curva de gasto: Q = {A:.3f} · (h − {H0:.3f})^{B:.3f}     R² = {R2:.4f}")
if H0_MANO is not None:
    print(f"Tu estimación de h0: {H0_MANO:.3f} m · diferencia con el ajuste: {100*(H0_MANO-H0):+.1f} cm")

af["Q_curva"] = gasto(af.nivel_m)
af["residuo_%"] = 100 * (af.caudal_m3s - af.Q_curva) / af.Q_curva

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
axs[0].plot(res.h0, res.R2, color="#1F6F8B"); axs[0].axvline(H0, color="#B3541A", ls="--")
axs[0].set_xlabel("h0 probado (m)"); axs[0].set_ylabel("R²"); axs[0].set_title("Búsqueda de h0", fontweight="bold")
x = np.log10(h - H0); axs[1].plot(x, np.log10(Q), "o", color="#12263A")
xx = np.linspace(x.min(), x.max(), 10); axs[1].plot(xx, np.log10(A) + B * xx, color="#1F6F8B", lw=2)
axs[1].set_xlabel("log (h − h0)"); axs[1].set_ylabel("log Q"); axs[1].set_title(f"Recta: pendiente b = {B:.2f}", fontweight="bold")
axs[2].axhline(0, color="k", lw=0.8); axs[2].bar(range(len(af)), af["residuo_%"], color="#B3541A")
axs[2].set_xlabel("Aforo (ordenado por nivel)"); axs[2].set_ylabel("Residuo (%)"); axs[2].set_title("Aforo − curva", fontweight="bold")
for a_ in axs: a_.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("salidas/P3_01_ajuste.png", dpi=150); plt.show()
print(f"Residuos: medio {af['residuo_%'].mean():+.1f} % · máximo absoluto {af['residuo_%'].abs().max():.1f} %")

## 4 · La curva de gasto y su zona de extrapolación

In [ ]:
h_max_aforo = h.max(); h_top = max(nv.nivel_m.max(), h_max_aforo) * 1.05
hh = np.linspace(H0 + 1e-3, h_top, 300)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axvspan(h_max_aforo, h_top, color="#B3541A", alpha=0.08)
ax.plot(hh[hh <= h_max_aforo], gasto(hh[hh <= h_max_aforo]), color="#1F6F8B", lw=3, label="Curva ajustada")
ax.plot(hh[hh >= h_max_aforo], gasto(hh[hh >= h_max_aforo]), color="#B3541A", lw=2.5, ls="--", label="Extrapolación")
ax.plot(h, Q, "o", color="#12263A", label="Aforos")
ax.set_xlabel("Nivel h (m)"); ax.set_ylabel("Caudal Q (m³/s)"); ax.legend(frameon=False); ax.grid(alpha=0.3)
ax.set_title(f"Curva de gasto · {ESTACION}\nQ = {A:.2f} (h − {H0:.3f})^{B:.2f}", fontweight="bold")
plt.tight_layout(); plt.savefig("salidas/P3_02_curva_gasto.png", dpi=150); plt.show()

## 5 · Del nivel diario al caudal diario

In [ ]:
nv["caudal_m3s"] = gasto(nv.nivel_m)
nv["extrapolado"] = nv.nivel_m > h_max_aforo
pct_dias = 100 * nv.extrapolado.mean()
pct_vol = 100 * nv.loc[nv.extrapolado, "caudal_m3s"].sum() / nv.caudal_m3s.sum()
print(f"Días con nivel por encima del aforo más alto ({h_max_aforo:.2f} m): {nv.extrapolado.sum()} ({pct_dias:.1f} %)")
print(f"Volumen que pasa en esos días: {pct_vol:.1f} % del total")
Qmed = nv.caudal_m3s.mean()
fig, axs = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [2.3, 1]})
axs[0].fill_between(nv.fecha, nv.caudal_m3s, color="#BFD9E3"); axs[0].plot(nv.fecha, nv.caudal_m3s, color="#1F6F8B", lw=0.8)
axs[0].plot(nv.fecha[nv.extrapolado], nv.caudal_m3s[nv.extrapolado], ".", color="#B3541A", ms=4, label="Extrapolado")
axs[0].axhline(Qmed, color="#B3541A", ls="--", lw=1.2, label=f"Caudal medio {Qmed:.2f} m³/s")
axs[0].set_ylabel("Caudal diario (m³/s)"); axs[0].legend(frameon=False); axs[0].set_title("Hidrograma diario", fontweight="bold")
mens = nv.groupby(nv.fecha.dt.month).caudal_m3s.mean()
axs[1].bar(mens.index, mens.values, color="#1F6F8B"); axs[1].set_xticks(range(1, 13))
axs[1].set_xticklabels(list("EFMAMJJASOND")); axs[1].set_ylabel("Caudal medio mensual (m³/s)")
axs[1].set_title("Régimen mensual", fontweight="bold")
for a_ in axs: a_.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("salidas/P3_03_hidrograma.png", dpi=150); plt.show()

## 6 · Curva de duración y caudales característicos
Se ordenan los caudales de mayor a menor y a cada puesto *m* se le asigna $p = m/(n+1)\times100$ (Weibull).
$Q_p$ es el caudal igualado o superado el *p* % del tiempo.

In [ ]:
qs = np.sort(nv.caudal_m3s.values)[::-1]
n = len(qs); p = np.arange(1, n + 1) / (n + 1) * 100
car = {f"Q{k}": float(np.interp(k, p, qs)) for k in (10, 50, 90, 95)}
pct_media = 100 * (nv.caudal_m3s >= Qmed).mean()
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.semilogy(p, qs, color="#1F6F8B", lw=3)
for k, q in car.items():
    pp = int(k[1:]); ax.plot([pp, pp], [qs.min(), q], color="#B3541A", ls="--", lw=1); ax.plot(pp, q, "o", color="#B3541A")
    ax.text(pp + 1, q * (0.80 if k == "Q95" else 1.12), f"{k} = {q:.2f}", color="#B3541A", va="top" if k == "Q95" else "baseline")
ax.axhline(Qmed, color="#12263A", lw=1, ls=":"); ax.text(60, Qmed * 1.1, f"caudal medio {Qmed:.2f} (superado el {pct_media:.0f} % del tiempo)", fontsize=9)
ax.set_xlabel("% del tiempo en que el caudal es igualado o superado"); ax.set_ylabel("Caudal (m³/s, escala log)")
ax.set_title(f"Curva de duración · {ESTACION}", fontweight="bold"); ax.set_xlim(0, 100); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.savefig("salidas/P3_04_curva_duracion.png", dpi=150); plt.show()

tabla = pd.DataFrame({
  "Indicador": ["Curva de gasto", "R² del ajuste", "Aforo más alto (m)", "Días extrapolados (%)", "Volumen extrapolado (%)",
                "Caudal medio (m³/s)", "% del tiempo que se supera el medio", "Q10 (m³/s)", "Q50 (m³/s)", "Q90 (m³/s)", "Q95 (m³/s)", "Q90 / Q50"],
  "Valor": [f"Q = {A:.3f} (h − {H0:.3f})^{B:.3f}", f"{R2:.4f}", f"{h_max_aforo:.2f}", f"{pct_dias:.1f}", f"{pct_vol:.1f}",
            f"{Qmed:.3f}", f"{pct_media:.0f}", f"{car['Q10']:.3f}", f"{car['Q50']:.3f}", f"{car['Q90']:.3f}", f"{car['Q95']:.3f}",
            f"{car['Q90']/car['Q50']:.2f}"]})
tabla.to_csv(f"salidas/resumen_curvas_{SLUG}.csv", index=False, encoding="utf-8-sig")
nv.to_csv(f"salidas/serie_caudal_diario_{SLUG}.csv", index=False, encoding="utf-8-sig")

# Expediente del proyecto integrador: mismo formato que el paso 2
origen = "datos propios" if USAR_MIS_DATOS else "datos de ejemplo del curso (no son de tu cuenca)"
fuente = f"P1_paso3 · {ESTACION} · {origen}"
exp = [("1.6", "Fuente de los datos de caudal", f"{ESTACION} · {origen}", "", "celda 1"),
       ("1.6", "Periodo de la serie", f"{nv.fecha.min():%Y-%m-%d} a {nv.fecha.max():%Y-%m-%d}", "", fuente),
       ("1.6", "Número de aforos", len(af), "n.º", fuente),
       ("1.6", "Curva de gasto a", round(A, 3), "—", fuente),
       ("1.6", "Curva de gasto h0", round(H0, 3), "m", fuente),
       ("1.6", "Curva de gasto b", round(B, 3), "—", fuente),
       ("1.6", "R² del ajuste", round(R2, 4), "—", fuente),
       ("1.6", "Aforo más alto", round(h_max_aforo, 2), "m", fuente),
       ("1.6", "Volumen extrapolado", round(pct_vol, 1), "%", fuente),
       ("1.6", "Caudal medio", round(Qmed, 3), "m³/s", fuente),
       ("1.6", "Tiempo en que se supera el caudal medio", round(pct_media), "%", fuente)]
exp += [("1.6", k, round(v, 3), "m³/s", fuente) for k, v in car.items()]
exp += [("1.6", "Q90 / Q50", round(car["Q90"] / car["Q50"], 2), "—", fuente)]
EXP = pd.DataFrame([("1",) + f for f in exp], columns=["capitulo", "apartado", "campo", "valor", "unidad", "fuente"])
EXP.to_csv(f"salidas/expediente_P3_{CUENCA}_{SLUG}.csv", index=False, encoding="utf-8-sig")
tabla

## 7 · Descargar resultados

In [ ]:
import shutil
paquete = shutil.make_archive(f"Practica1_Paso3_{SLUG}", "zip", "salidas")
print("✅ Paquete:", paquete)
try:
    from google.colab import files; files.download(paquete)
except Exception:
    print("Descárgalo desde el panel de archivos de Colab.")

---
## Para la ficha de tu cuenca (cinco líneas cada una)
1. ¿Qué porcentaje del volumen salió de niveles por encima del aforo más alto? ¿Qué harías para reducir esa incertidumbre?
2. Compara el caudal medio con Q50. ¿Qué caudal usarías para diseñar un abastecimiento de agua y por qué?
3. Con Q90 / Q50, ¿el río es regular o estacional? Relaciónalo con la forma y el relieve de tu cuenca (paso 2).